In [1]:
# library imports
import rasterio, geopandas as gpd, numpy as np, pandas as pd
from pathlib import Path
from rasterio.mask import mask

In [ ]:
class Banana_Health:

    def __init__(self, in_count_geojson_path:Path, in_ndvi_tiff_path:Path, in_ndre_tiff_path:Path, in_tvi_tiff_path:Path):
        """ in_count_geojson: Path of count point geojson,
            in_ndvi_tiff_path: Path of NDVI orthomosaic geotiff image,
            in_ndre_tiff_path: NDRE index dir,
            in_tvi_tiff_path: Triangular Vegetative Index dir. """
        
        # organize all paths
        self.paths = ["point_path", "ndvi_image_path", "ndre_path", "tvi_path"]
        inputs = [in_count_geojson_path, in_ndvi_tiff_path, in_ndre_tiff_path, in_tvi_tiff_path]

        # infer class attribute
        for attr, input_path in zip(self.paths, inputs):
            setattr(self, attr, Path(input_path))
        
        # input directory validation
        if not self.point_path.exists() and self.ndvi_image_path.exists():
            raise FileNotFoundError(f"one or both directories '{self.point_path}', and '{self.ndvi_image_path}' does not exist.")
            return
        
        # create a health_dir to save health classification output
        self.health_path = self.point_path.resolve().parent.joinpath('Banana_health_geo_output')
        self.health_path.mkdir(parents=True, exist_ok=True)
    

    # read points and buffer geometry    
    def points_buffer(self, point_path):
        point_gdf = gpd.read_file(point_path)

        # buffer each point by 1 meter
        point_gdf['geometry_buf'] = point_gdf.geometry.buffer(1)

        return point_gdf
    

    # extract image indices using mask
    def image_values(self, image, row_geometry):

        # mask the raster with the polygon
        out_image, out_transform = mask(image, [row_geometry], crop=True)

        # get the first band (if multi-band)
        out_image = out_image[0]

        # extract the DN values within the polygon
        dn_values = out_image.flatten()
        pos_dn_values = dn_values[dn_values > 0]

        # get the coordinates of the pixels
        rows, cols = np.indices(out_image.shape)
        xs, ys = rasterio.transform.xy(out_transform, rows, cols)

        # convert to arrays
        xs = np.array(xs)
        ys = np.array(ys)

        return dn_values, pos_dn_values, xs, ys
    

    # get pixel distance from centroid
    def dn_distance(self, row_geometry, all_x, all_y, dn_val):

        # centroid of the polygon
        centroid = row_geometry.centroid

        # compute distance from each pixel to the centroid
        distances = np.sqrt((all_x - centroid.x)**2 + (all_y - centroid.y)**2)

        # flatten arrays
        distances = distances.flatten()

        # use +dn_values to filter dn_distances
        pos_distances = distances[dn_val > 0]

        return pos_distances
    
    
    # locate central pixels (pixels within distance threshold)
    def percentile_filter(self, dn_distance, percentile_num, dn_vals):
        
        # set limit to a percentile of closest pixels
        threshold_distance = np.percentile(dn_distance, percentile_num) 
        
        # filter threshold pixels
        central_pixs = dn_vals[dn_distance <= threshold_distance]
        
        return central_pixs
    
    
    # compute weighted mean based on proximity to the centroid
    def mean_dn(self, central_pxls):

        # Adding a small value to avoid division by zero and normalize
        weights = 1 / (central_pxls + 1e-10)  
        normalized_weights = weights / np.sum(weights)

        # avearge dn
        weighted_mean_dn = np.average(central_pxls.flatten(), weights=normalized_weights.flatten())
        weighted_mean_dn = round(weighted_mean_dn,7)

        if weighted_mean_dn < 0:
            print(f"Weighted Mean DN value (central pixels): {weighted_mean_dn}")

        return weighted_mean_dn
    
    
    # compute vi_index
    def all_VIs (self, buf_pt_gdf):

        # Iterate over attribute names
        for attr in self.paths[1:]:

            # Retrieve the actual Path object from the attribute
            full_path = getattr(self, attr)

            # get the name of index
            idx_name = full_path.name

            # iterate the index dir for image file
            for img_file in full_path.iterdir():

                # read image file
                with rasterio.open(img_file) as src_img:

                    # iterate buffer and extract index
                    for idx, row in buf_pt_gdf.iterrows():
                        
                        # extract image values
                        dn_values, pos_dn, xs, ys = self.image_values(src_img, row['geometry_buf'])

                        # get distance to image values 
                        pos_distances = self.dn_distance(row['geometry_buf'], xs, ys, dn_values)
        
                        # filter image values of top 50% elevation
                        central_pixels = self.percentile_filter(pos_distances, 50, pos_dn)
        
                        # compute the mean of filtered image values
                        weighted_mean_dn = self.mean_dn(central_pixels)

                        # assign mean value to point location
                        buf_pt_gdf.at[idx, idx_name] = weighted_mean_dn

            print(f"{idx_name} index completed")

        buf_pt_gdf = buf_pt_gdf.drop(columns=['geometry_buf'])

        return buf_pt_gdf
    
    
    def range_vals(self, pt_gdf, class_type):
        # mean and std
        mean_val = round(pt_gdf[class_type].mean(),2)
        std_val = round(pt_gdf[class_type].std(),2)
        
        # range category set
        return {
            "Very Unhealthy": (round(pt_gdf[class_type].min(),2) - 0.01, mean_val - round(std_val,2)),
            "Unhealthy": (mean_val - round(std_val,2), mean_val),
            "Healthy": (mean_val, mean_val + round(std_val,2)),
            "Very Healthy": (mean_val + round(std_val,2), pt_gdf[class_type].max() + 0.01),
        }
    
    
    # define health classification function
    def classify_health(self, pt_gdf, value, class_type):
        health_ranges = {
            "NDRE": self.range_vals(pt_gdf, "NDRE"),
            "TVI": self.range_vals(pt_gdf, "TVI"),
            "NDVI": {
                "Very Unhealthy": (0.3, 0.45),
                "Unhealthy": (0.45, 0.50),
                "Healthy": (0.50, 0.83),
                "Very Healthy": (0.83, 0.9),
            },
            "all_scores": {
                "Very Unhealthy": (0, 1),
                "Unhealthy": (1, 2),
                "Healthy": (2, 3),
                "Very Healthy": (3, 5),
            },
        }
        for category, (low, high) in health_ranges.get(class_type, {}).items():
            if low <= value < high:
                return category
        return "Unknown"
    

    # define and applly health score
    def health_score(self, vi_health):
        # map dictionary
        health_scores = {
            'Very Healthy': 4,
            'Healthy': 3,
            'Unhealthy': 2,
            'Very Unhealthy': 1
        }
        # apply mapping
        return vi_health.map(health_scores)
    
    
    # calculate combined health value
    def Health_val (self, pt_gdf):

        # Define Kappa values
        kappa_weights = [0.3498, 0.3416, 0.3086] # [0.85, 0.83, 0.75]

        # Apply VI health classification efficiently
        for index in ["NDVI", "NDRE", "TVI"]:
            pt_gdf[f"{index}_score"] = self.health_score(pt_gdf[f"{index}_Health_class"])

        # Compute weighted average
        pt_gdf["all_scores"] = (
            (pt_gdf["TVI_score"] * kappa_weights[0]) +
            (pt_gdf["NDRE_score"] * kappa_weights[1]) +
            (pt_gdf["NDVI_score"] * kappa_weights[2])
        )

        return pt_gdf
    

    # execute process
    def process(self):
         
         # read csv's as df
        for point_file in self.point_path.iterdir():

            # get the name of plantation
            name_prefix = point_file.stem.split('_')[0]
            
            # buffer the crown points
            crown_gdf = self.points_buffer(point_file)

            # generate the vegetative indices
            points_gdf = self.all_VIs(crown_gdf)

            # apply VI health classification
            for index in ["NDVI", "NDRE", "TVI"]:
                points_gdf[f"{index}_Health_class"] = points_gdf[index].apply(lambda val: self.classify_health(val, index))

            # get all health classes
            points_gdf = self.Health_val(points_gdf)
            points_gdf["Health status"] = points_gdf["all_scores"].apply(lambda val: self.classify_health(val,"all_scores"))

        # print(f'all scores min:{points_gdf['all_scores'].min()}, max: {points_gdf['all_scores'].max()}, mean: {points_gdf['all_scores'].mean()}, std: {points_gdf['all_scores'].std()}')

        # save final table
        out_path = f'{self.health_path}/{name_prefix}_health_summary.geojson'

        # save the health_summary_gdf as geojson file
        points_gdf.to_file(out_path, driver='GeoJSON')

        # indicate health status completion
        print(f'{out_path} successfully completed.')


# Data repository
in_point_dir = '.../Project_repo/Banana_health_geo_output/' # point_count dir
in_NDVI_dir = '.../Project_repo/banana_health/NDVI/' # NDVI dir
in_NDRE_dir = '.../Project_repo/banana_health/NDRE/' # NDRE dir
in_TVI_dir = '.../Project_repo/health/banana_health/TVI/' # TVI dir

# Implementation
NDVI_processing = Banana_Health(in_point_dir, in_NDVI_dir, in_NDRE_dir, in_TVI_dir)
NDVI_processing.process()

NDVI index completed
NDRE index completed
TVI index completed
NDVI min:0.4465157091617584, max: 0.883977472782135, mean: 0.7823956608772278, std: 0.0521509125828743
NDRE min:0.1349678933620453, max: 0.34793341159820557, mean: 0.253755122423172, std: 0.023123331367969513
TVI min:0.9871150255203247, max: 1.17705500125885, mean: 1.13507878780365, std: 0.0212030541151762
all scores min:1.3086, max: 4.0, mean: 2.6987642382812496, std: 0.6400663529782474


,Plant_id,long,lat,block_id,elevation_status,Surface_value,status,geometry,NDVI,NDRE,TVI,NDVI_Health_class,NDRE_Health_class,TVI_Health_class,NDVI_score,NDRE_score,TVI_score,all_scores,Health status
0,1,177384.969461,692168.955959,1,standing,52.469948,Sucker,POINT (177384.969 692168.956),0.762946,0.242078,1.130791,Healthy,Unhealthy,Unhealthy,3.0,2,2.0,2.3086,Healthy
1,2,177384.902052,692170.296067,1,standing,52.506973,Parent,POINT (177384.902 692170.296),0.741367,0.234705,1.117048,Healthy,Unhealthy,Very Unhealthy,3.0,2,1.0,1.9588,Unhealthy
2,3,177383.301593,692172.241822,1,standing,52.923218,Parent,POINT (177383.302 692172.242),0.724556,0.239044,1.115413,Healthy,Unhealthy,Very Unhealthy,3.0,2,1.0,1.9588,Unhealthy
3,4,177381.071945,692173.970937,1,standing,52.440468,Parent,POINT (177381.072 692173.971),0.750725,0.223774,1.125665,Healthy,Very Unhealthy,Unhealthy,3.0,1,2.0,1.9670,Unhealthy
4,5,177383.531365,692174.442395,1,standing,53.207237,Parent,POINT (177383.531 692174.442),0.789792,0.271180,1.138522,Healthy,Very Healthy,Unhealthy,3.0,4,2.0,2.9918,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10237,10238,177605.365378,692152.546245,9,standing,48.704887,Parent,POINT (177605.365 692152.546),0.826425,0.291231,1.152540,Healthy,Very Healthy,Healthy,3.0,4,3.0,3.3416,Very Healthy
10238,10239,177605.064662,692149.258422,9,standing,48.820580,Parent,POINT (177605.065 692149.258),0.795347,0.261373,1.143126,Healthy,Healthy,Healthy,3.0,3,3.0,3.0000,Very Healthy
10239,10240,177616.070849,692161.888473,9,standing,48.035965,Parent,POINT (177616.071 692161.888),0.819744,0.267481,1.150063,Healthy,Healthy,Healthy,3.0,3,3.0,3.0000,Very Healthy
10240,10241,177624.370596,692161.848377,9,standing,49.458000,Parent,POINT (177624.371 692161.848),0.855894,0.268500,1.164790,Very Healthy,Healthy,Very Healthy,4.0,3,4.0,3.6584,Very Healthy
